# 04 — Resolution and Capacity Sensitivity

**Pipeline stage:** aggregating single-country evaluations (notebook 03) across all 19 countries, four
resolutions, and two aggregation schemes, and asking two of the paper's remaining research questions:

- **RQ3:** does a country's wind/solar mix explain differences in weather's added value?
- **RQ4:** how much do capacity weighting and grid resolution actually matter, and what do they cost
  computationally?

`scripts/summarize_spatial_resolution.py` aggregates per-country/per-resolution/per-scheme results
into summary tables (correlation of gain vs. wind-minus-solar share, resolution-drift vs. the 0.25°
reference, compute-cost summary, and a country-size sensitivity check). `scripts/bootstrap_post_covid_all_models.py`
adds confidence intervals via block-bootstrap resampling, since single-country gains are noisy point
estimates otherwise.

In [1]:
!python ../scripts/summarize_spatial_resolution.py --help

usage: summarize_spatial_resolution.py [-h] [--results-dir RESULTS_DIR]
                                       [--country-summary COUNTRY_SUMMARY]
                                       [--figure-dir FIGURE_DIR]

options:
  -h, --help            show this help message and exit
  --results-dir RESULTS_DIR
  --country-summary COUNTRY_SUMMARY
  --figure-dir FIGURE_DIR


In [2]:
!python ../scripts/bootstrap_post_covid_all_models.py --help

usage: bootstrap_post_covid_all_models.py [-h]
                                          [--codes {dk,ie,nl,pt,gr,be,lt,hr,bg,lv,si,rs,sk,de,es,fr,at,cz,ro} [{dk,ie,nl,pt,gr,be,lt,hr,bg,lv,si,rs,sk,de,es,fr,at,cz,ro} ...]]
                                          [--models {LogReg,RandForest,GradientBoosting,LightGBM} [{LogReg,RandForest,GradientBoosting,LightGBM} ...]]
                                          [--n-resamples N_RESAMPLES]
                                          [--confidence CONFIDENCE]
                                          [--seed SEED] [--workers WORKERS]
                                          [--backend {process,thread}]
                                          [--split SPLIT]
                                          [--block-mode {iso_week,fixed_days}]
                                          [--block-days BLOCK_DAYS]
                                          [--energy-dir ENERGY_DIR]
                                          [--weather-dir WEATHER_DIR

## Why block-bootstrap, not ordinary bootstrap?

Hourly renewable-share data is strongly autocorrelated — an hour's value is highly predictable from
the previous hour's. Resampling individual hours independently (an ordinary bootstrap) would
understate uncertainty, because it pretends each hour is an independent observation when it isn't.
Block-bootstrapping resamples *whole ISO weeks* (or fixed multi-day blocks) instead, preserving the
within-block autocorrelation structure, then refits nothing — it resamples predictions on the already
fixed, already-trained model's test-set output, which is why `bootstrap_post_covid_all_models.py` is
far cheaper than the legacy `bootstrap_core.py`, which re-fits the model on every resample.

## The compute/resolution tradeoff, demonstrated on toy cell counts

`summarize_spatial_resolution.py` reports workload reduction in **grid-point hours**
($H_r = \sum_{\text{countries}} N_r \times T$) — the number of weather cells processed, times the
number of hourly timesteps. Because coarsening a 0.25° grid to 1° collapses each 4×4 block of native
cells into one, the cell count $N_r$ alone drops by a predictable factor, independent of how many
countries or hours are involved:

In [3]:
resolutions_deg = [0.25, 0.5, 1.0, 2.0]
# Cells per side halves (so cells per 2D block quarters) each time resolution doubles,
# relative to the 0.25 deg reference.
reference = resolutions_deg[0]
for res in resolutions_deg:
    block_side = res / reference          # e.g. 1.0 / 0.25 = 4 -> a 4x4 = 16-cell block
    cells_per_block = block_side ** 2
    reduction_pct = 100 * (1 - 1 / cells_per_block) if cells_per_block > 1 else 0.0
    print(f"{res:>4.2f} deg:  block = {block_side:.0f}x{block_side:.0f} = {cells_per_block:>5.0f} "
          f"native cells collapsed into 1  ->  {reduction_pct:5.1f}% fewer cells than 0.25 deg")

0.25 deg:  block = 1x1 =     1 native cells collapsed into 1  ->    0.0% fewer cells than 0.25 deg
0.50 deg:  block = 2x2 =     4 native cells collapsed into 1  ->   75.0% fewer cells than 0.25 deg
1.00 deg:  block = 4x4 =    16 native cells collapsed into 1  ->   93.8% fewer cells than 0.25 deg
2.00 deg:  block = 8x8 =    64 native cells collapsed into 1  ->   98.4% fewer cells than 0.25 deg


This matches the paper's headline number: at 1°, `4x4 = 16` native cells collapse into one, so only
`1/16 = 6.25%` of the original cells remain — a **93.75%** reduction, consistent with the paper's
reported "approximately 93%". The paper's actual finding is that this compute saving comes at *almost
no accuracy cost* — that comparison needs the real per-resolution model results, produced by running
notebook 03's evaluation at each resolution and comparing with `summarize_spatial_resolution.py`.

**DATA CELL — not run here.** A real end-to-end resolution sweep for one era:

```bash
python scripts/run_era_spatial_resolution.py --eras post --codes de fr es --resolutions 0.25 0.5 1.0 2.0 \
    --schemes capacity uniform --steps fetch inputs capacity stage cache
python scripts/summarize_spatial_resolution.py --results-dir results/post_covid_spatial_resolution
python scripts/bootstrap_post_covid_all_models.py --codes de fr es
```

Note: as documented in the repo README, three of `run_era_spatial_resolution.py`'s later steps
(`audit_spatial_pipeline_data.py`, `run_spatial_resolution_ladder.py`, `compare_era_spatial_resolution.py`)
are not yet in this repo — the `--steps` above are limited to what's currently available.

**Next:** [05 — Figures and Bootstrap CI](05_figures_and_bootstrap_ci.ipynb) turns these summary tables
into the actual figures used in the paper.